In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from scipy.stats import pearsonr, spearmanr


from utils_correlations import replace_greek, inverse_replace_greek, get_gene_conversion_dict
from utils_name_maps import cytokine_name_map

In [ ]:
# Data obtained from: https://pmc.ncbi.nlm.nih.gov/articles/PMC5784859 
# NIHMS921103-supplement-supp_table1.xlsx

In [ ]:
FIGDIR = "./figures_new"

In [ ]:
human_to_mouse_dict, mouse_to_human_dict = get_gene_conversion_dict()

In [ ]:
def get_data_kang(celltype: str | None = None):
    rename_celltype_map = {
        "nk": "NK",
        "cd14": "CD14_Mono",
        "cd16 mono": "CD16_Mono",
        "dc": "cDC",
        "cd4": "CD4_T_cell",
        "cd8": "CD8_T_cell",
    }
    inverse_rename_celltype_map = {rename_celltype_map[key]: key for key in rename_celltype_map.keys()}
    if celltype is None:
        xl = pd.ExcelFile("/Users/soeren.becker/repos/HumanCytokineDict/data/Kang_et_al/NIHMS921103-supplement-supp_table1.xlsx")
        celltypes = xl.sheet_names
    else:
        celltypes = [inverse_rename_celltype_map[celltype]]

    dfs = []
    for celltype in tqdm(celltypes):
        df = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/Kang_et_al/NIHMS921103-supplement-supp_table1.xlsx", sheet_name=celltype)
        df.loc[:, "celltype"] = rename_celltype_map.get(celltype, None)
        dfs.append(df)
    df_kang = pd.concat(dfs, axis=0, ignore_index=True)
    df_kang.rename({"Unnamed: 0": "ensemble_id", "log2FoldChange": "log_fc", "featureData.symbol": "gene", "padj": "adj_p_value"}, axis=1, inplace=True)
    df_kang.loc[:, "cytokine"] = "IFN-beta"
    df_kang = df_kang.loc[~df_kang.celltype.isna()]
    rename_celltype_map
    return df_kang


def get_human_data(
    celltype: str | None = None,
    cytokine: str | None = None,
    hcd: pd.DataFrame | None = None,
)-> pd.DataFrame:

    if hcd is None:
        hcd = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_with_blacklisted_genes_removed.csv", index_col=0)
    hcd = hcd.loc[hcd.celltype == celltype]        
    if cytokine is not None:
        hcd = hcd.loc[hcd.cytokine == cytokine]
    return hcd


def get_data_mouse(
    celltype: str | None = None, 
    cytokine: str | None = None, 
    mouse_to_human_dict: dict | None = None,
    subset_to_homologs: bool = True,
) -> pd.DataFrame:
    
    mcd = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/41586_2023_6816_MOESM5_ESM.xlsx", sheet_name=celltype)
    mcd.loc[:, "cytokine"]  = mcd.Cytokine_Str.map(cytokine_name_map)
    mcd = mcd.loc[~mcd.cytokine.isna()]
    if cytokine is not None:
        mcd = mcd.loc[mcd.cytokine == cytokine]
    mcd.rename({"Gene": "gene", "Celltype_Str": "celltype", "Avg_log2FC": "log_fc"}, axis=1, inplace=True)
    mcd.loc[:, "gene_mouse"] = mcd.gene.copy()
    
    if mouse_to_human_dict is not None:
        mcd.loc[:, "gene"] = mcd.loc[:, "gene"].map(mouse_to_human_dict)
        if subset_to_homologs:
            mcd = mcd.loc[~mcd.gene.isna()]
    return mcd

def jaccard(x, y):
    intersection = np.intersect1d(x, y)
    union = np.union1d(x, y)
    return len(intersection) / len(union)

In [ ]:
homologs = list(mouse_to_human_dict.values())
print(len(homologs))

In [ ]:
# load once
hcd_all = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_with_blacklisted_genes_removed.csv", index_col=0)
hcd_all = hcd_all.loc[hcd_all.cytokine == "IFN-beta"]
hcd_all = hcd_all.loc[(hcd_all.log_fc.abs() > 1) & (hcd_all.num_DE_pbs_wells > 3) & (hcd_all.adj_p_value < 0.05)]
# hcd_all = hcd_all.loc[(hcd_all.log_fc.abs() > 1) & (hcd_all.num_DE_pbs_wells > 3) & (hcd_all.adj_p_value < 0.05)]

In [ ]:
celltype_combinations = (
    # kang, hcd, mcd
    ("CD14_Mono", "CD14_Mono", "Monocyte"),
    ("CD16_Mono", "CD16_Mono", "Monocyte"),
    ("CD4_T_cell", "CD4_T_cell", "T_cell_CD4"),
    ("CD8_T_cell", "CD8_T_cell", "T_cell_CD8"),
    ("cDC", "cDC", "cDC1"),
    ("cDC", "cDC", "cDC2"),
    ("NK_CD56hi", "NK", "NK_cell"),
    ("NK_CD56low", "NK", "NK_cell"),
)

In [ ]:
scores = pd.DataFrame()

for celltype_combo in celltype_combinations:
    celltype_hcd, celltype_kang, celltype_mcd = celltype_combo
        
    df_hcd = get_human_data(celltype = celltype_hcd, hcd = hcd_all)
    df_kang = get_data_kang(celltype_kang)
    df_mcd = get_data_mouse(celltype=celltype_mcd, cytokine="IFN-beta", mouse_to_human_dict=mouse_to_human_dict, subset_to_homologs=True)
    df_mcd = df_mcd.loc[df_mcd.log_fc.abs() > 1]
    shared_genes_hcd_kang = np.intersect1d(df_hcd.gene.astype(str), df_kang.gene.astype(str))
    shared_genes_hcd_mcd = np.intersect1d(df_hcd.gene.astype(str), df_mcd.gene.astype(str))
    shared_genes_kang_mcd = np.intersect1d(df_mcd.gene.astype(str), df_kang.gene.astype(str))
    shared_genes = np.intersect1d(shared_genes_hcd_kang, shared_genes_kang_mcd)
    lfc_hcd = df_hcd.set_index("gene").loc[shared_genes].log_fc
    lfc_mcd = df_mcd.set_index("gene").loc[shared_genes].log_fc
    lfc_kang = df_kang.set_index("gene").loc[shared_genes].log_fc
    
    row_idx = scores.shape[0]
    scores.loc[row_idx, "comparison"] = "hcd_kang"
    scores.loc[row_idx, "celltype_combo"] = "__".join(celltype_combo)
    scores.loc[row_idx, "num_shared_genes"] = len(shared_genes_hcd_kang)
    scores.loc[row_idx, "num_genes_1"] = df_hcd.shape[0]
    scores.loc[row_idx, "num_genes_1_homologs"] = df_hcd.loc[df_hcd.gene.isin(homologs)].shape[0]
    scores.loc[row_idx, "num_genes_2"] = df_kang.shape[0]
    scores.loc[row_idx, "num_genes_2_homologs"] = df_kang.loc[df_kang.gene.isin(homologs)].shape[0]
    scores.loc[row_idx, "jaccard"] = jaccard(df_hcd.gene.astype(str), df_kang.gene.astype(str))
    scores.loc[row_idx, "pearson"] = pearsonr(lfc_hcd, lfc_kang)[0]
    scores.loc[row_idx, "spearman"] = spearmanr(lfc_hcd, lfc_kang)[0]
    
    row_idx = scores.shape[0]
    scores.loc[row_idx, "comparison"] = "hcd_mcd"
    scores.loc[row_idx, "celltype_combo"] = "__".join(celltype_combo)
    scores.loc[row_idx, "num_shared_genes"] = len(shared_genes_hcd_mcd)
    scores.loc[row_idx, "num_genes_1"] = df_hcd.shape[0]
    scores.loc[row_idx, "num_genes_1_homologs"] = df_hcd.loc[df_hcd.gene.isin(homologs)].shape[0]
    scores.loc[row_idx, "num_genes_2"] = df_mcd.shape[0]
    scores.loc[row_idx, "num_genes_2_homologs"] = df_mcd.shape[0]
    scores.loc[row_idx, "jaccard"] = jaccard(df_hcd.gene.astype(str), df_mcd.gene.astype(str))
    scores.loc[row_idx, "pearson"] = pearsonr(lfc_hcd, lfc_mcd)[0]
    scores.loc[row_idx, "spearman"] = spearmanr(lfc_hcd, lfc_mcd)[0]
    
    row_idx = scores.shape[0]
    scores.loc[row_idx, "comparison"] = "kang_mcd"
    scores.loc[row_idx, "celltype_combo"] = "__".join(celltype_combo)
    scores.loc[row_idx, "num_shared_genes"] = len(shared_genes_kang_mcd)
    scores.loc[row_idx, "num_genes_1"] = df_kang.shape[0]
    scores.loc[row_idx, "num_genes_1_homologs"] = df_kang.loc[df_kang.gene.isin(homologs)].shape[0]
    scores.loc[row_idx, "num_genes_2"] = df_mcd.shape[0]
    scores.loc[row_idx, "num_genes_2_homologs"] = df_mcd.shape[0]
    scores.loc[row_idx, "jaccard"] = jaccard(df_kang.gene.astype(str), df_mcd.gene.astype(str))
    scores.loc[row_idx, "pearson"] = pearsonr(lfc_kang, lfc_mcd)[0]
    scores.loc[row_idx, "spearman"] = spearmanr(lfc_kang, lfc_mcd)[0]

In [ ]:
scores.loc[:, "frac_shared_genes_1"] = scores.loc[:, "num_shared_genes"] / scores.loc[:, "num_genes_1"]
scores.loc[:, "frac_shared_genes_2"] = scores.loc[:, "num_shared_genes"] / scores.loc[:, "num_genes_2"]

In [ ]:
df_pivot = scores.pivot(
    index="comparison",
    columns="celltype_combo",
    values="pearson"
)
order_y = ["hcd_kang", "hcd_mcd", "kang_mcd"]
order_x = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot.loc[order_y, order_x]
fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.heatmap(
    df_pivot,
    square=True,
    cmap="RdBu_r",
    center=0,
    vmin=0,
    vmax=1,
    annot=False,
    fmt=".1f",
    annot_kws = {"fontsize": 6, "family":"Arial"},
    cbar=False,
    linewidth=0.5,
    linecolor="white",
    ax=ax,
)
plt.xlabel("")
plt.ylabel("")

def clean_comparison_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_comparison_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        # "hcd_kang": "HCD vs Kang et al.",
        # "hcd_kang": "HCD vs Kang",
        "hcd_kang": "H vs K",
        # "hcd_mcd": "HCD vs MCD",
        "hcd_mcd": "H vs M",
        # "kang_mcd": "MCD vs. Kang et al.",
        # "kang_mcd": "MCD vs Kang",
        "kang_mcd": "M vs K",
    }
    return name_map[x]

def clean_celltype_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_celltype_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono",
        'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono",
        'CD4_T_cell__CD4_T_cell__T_cell_CD4': "CD4 T cell",
        'CD8_T_cell__CD8_T_cell__T_cell_CD8': "CD8 T cell", 
        'NK_CD56hi__NK__NK_cell': "NK CD56hi",
        'NK_CD56low__NK__NK_cell': "NK CD56low", 
        'cDC__cDC__cDC1': "cDC1", 
        'cDC__cDC__cDC2': "cDC2", 
    }
    return name_map[x]
    
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(df_pivot.index), fontsize=6, family="Arial", ha="left")
ax.tick_params(axis='y', which='major', pad=20)


plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.title(r"$R$", fontsize=6, family="Arial")
plt.tick_params(length=0)
plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_pearson.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()
display(df_pivot)
print(df_pivot.min().min(), df_pivot.max().max())

In [ ]:
df_pivot.min(axis=1)

In [ ]:
df_pivot.max(axis=1)

In [ ]:
df_pivot.median(axis=1)

In [ ]:
sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=mcolors.Normalize(vmin=-1, vmax=1))
sm.set_array(np.array([]))

fig, ax = plt.subplots(1, 1, figsize=(0.1, 0.45))
cbar = plt.colorbar(sm, cax=ax, orientation="vertical")

# cbar.set_label("# Genes", rotation=0, labelpad=-3, family='Arial', fontsize=6)
# cbar.ax.xaxis.set_ticks_position('top')
# cbar.ax.xaxis.set_label_position('top')

cbar.ax.tick_params(labelsize=6)
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)
cbar.outline.set_visible(False)
cbar.ax.set_yticks([df_pivot.min().min(), df_pivot.max().max()], [f"{df_pivot.min().min():.2f}", f"{df_pivot.max().max():.2f}"], family="Arial", fontsize=6)
cbar.ax.set_ylim(df_pivot.min().min(), df_pivot.max().max())
cbar.ax.tick_params(length=0)

for label in cbar.ax.get_yticklabels():
    label.set_fontfamily('Arial')

filename = f"heatmap_comparison_kang_et_al_pearson_colorbar_vertical.svg"
plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
dfs = []
_df = scores.loc[scores.comparison == "hcd_kang"].groupby("celltype_combo").num_genes_1_homologs.mean().reset_index().rename({"num_genes_1_homologs": "num_genes"}, axis=1)
_df.loc[:, "study"] = "hcd"
dfs.append(_df)
_df = scores.loc[scores.comparison == "hcd_kang"].groupby("celltype_combo").num_genes_2_homologs.mean().reset_index().rename({"num_genes_2_homologs": "num_genes"}, axis=1)
_df.loc[:, "study"] = "kang"
dfs.append(_df)
_df = scores.loc[scores.comparison == "hcd_mcd"].groupby("celltype_combo").num_genes_2_homologs.mean().reset_index().rename({"num_genes_2_homologs": "num_genes"}, axis=1)
_df.loc[:, "study"] = "mcd"
dfs.append(_df)
df_gene_count = pd.concat(dfs, axis=0)
df_pivot = df_gene_count.pivot(
    index="study",
    columns="celltype_combo",
    values="num_genes"
)

df_pivot = df_pivot.loc[:, order_x]

fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.heatmap(
    df_pivot,
    square=True,
    cmap="Greens",
    annot_kws = {"fontsize": 6, "family":"Arial"},
    cbar=False,
    linewidth=0.5,
    linecolor="white",
)
plt.xlabel("")
plt.ylabel("")

plt.yticks(0.5+np.arange(df_pivot.shape[0]), df_pivot.index, fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.title(r"# genes w. |lfc| > 1", fontsize=6, family="Arial")
plt.tick_params(length=0)
# plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_pearson.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
mouse_color = "#1B9E77"
human_color = "#7570B3"
kang_color = "#C08081"

label_color_map = {
    "mcd": mouse_color,
    "hcd": human_color,
    "kang": kang_color,
}

fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.barplot(
    data=df_gene_count,
    x="celltype_combo",
    y="num_genes",
    hue="study",
    order=order_x,
    ax=ax,
    legend=False,
    palette=label_color_map,
    dodge=True,
)
plt.xlabel("")
plt.ylabel("# genes", fontsize=6, family="Arial")
yticks = [10, 100, 1000]
plt.yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
plt.yscale("log")
plt.xticks(np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial", rotation=90)
ax.spines[['right', 'top']].set_visible(False)
ax.grid(axis="y")
ax.set_axisbelow(True)
y_margin=0.05
ax.margins(y=y_margin)
plt.savefig(os.path.join(FIGDIR, f"barplot_comparison_kang_et_al_num_large_lfcs.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
from scipy.stats import kendalltau
df_hk = pd.merge(df_gene_count.loc[df_gene_count.study == "hcd"], df_gene_count.loc[df_gene_count.study == "kang"], on="celltype_combo")
print(pearsonr(df_hk.num_genes_x, df_hk.num_genes_y)[0], spearmanr(df_hk.num_genes_x, df_hk.num_genes_y)[0], kendalltau(df_hk.num_genes_x, df_hk.num_genes_y).statistic)
diff_hk = abs(df_hk.num_genes_x - df_hk.num_genes_y)
print(diff_hk.median(), diff_hk.mean(), diff_hk.min(), diff_hk.max())

In [ ]:
df_hm = pd.merge(df_gene_count.loc[df_gene_count.study == "hcd"], df_gene_count.loc[df_gene_count.study == "mcd"], on="celltype_combo")
pearsonr(df_hm.num_genes_x, df_hm.num_genes_y)[0], spearmanr(df_hm.num_genes_x, df_hm.num_genes_y)[0], kendalltau(df_hm.num_genes_x, df_hm.num_genes_y).statistic
diff_hm = abs(df_hm.num_genes_x - df_hm.num_genes_y)
print(diff_hm.median(), diff_hm.mean(), diff_hm.min(), diff_hm.max())

In [ ]:
df_mk = pd.merge(df_gene_count.loc[df_gene_count.study == "kang"], df_gene_count.loc[df_gene_count.study == "mcd"], on="celltype_combo")
pearsonr(df_mk.num_genes_x, df_mk.num_genes_y)[0], spearmanr(df_mk.num_genes_x, df_mk.num_genes_y)[0], kendalltau(df_mk.num_genes_x, df_mk.num_genes_y).statistic
diff_mk = abs(df_mk.num_genes_x - df_mk.num_genes_y)
print(diff_mk.median(), diff_mk.mean(), diff_mk.min(), diff_mk.max())

In [ ]:
label_color_map = {
    "Mouse cytokine dict.": mouse_color,
    "Human cytokine dict.": human_color,
    "Kang et al.": kang_color,
}

handles = []
labels = []

for label, color in label_color_map.items():
    patch = mpatches.Patch(color=color, label=label)
    handles.append(patch)
    labels.append(label)

fig, ax = plt.subplots(figsize=(0.5, 0.3))
legend = ax.legend(handles, labels, loc='center', frameon=False, fontsize=6, ncol=1) 

for text in legend.get_texts():
    text.set_fontfamily('Arial')

ax.axis('off')
plt.savefig(os.path.join(FIGDIR, f"barplot_comparison_kang_et_al_legend.svg"), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_pivot = scores.pivot(
    index="comparison",
    columns="celltype_combo",
    values="spearman"
)
order_y = ["hcd_kang", "kang_mcd", "hcd_mcd"]
order_x = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.heatmap(
    df_pivot,
    square=True,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    # annot=True,
    annot_kws = {"fontsize": 6, "family":"Arial"},
    cbar=False,
    linewidth=0.5,
    linecolor="white",
)
plt.xlabel("")
plt.ylabel("")
    
# plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(ax.get_yticklabels()), fontsize=6, family="Arial")
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(df_pivot.index), fontsize=6, family="Arial")
# plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(ax.get_xticklabels()), fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.tick_params(length=0)
plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_spearman.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
df_pivot.min(axis=1)

In [ ]:
df_pivot.max(axis=1)

In [ ]:
ctc = "CD14_Mono__CD14_Mono__Monocyte"
overlaps = []

for ctc in scores.celltype_combo.unique():
    df_overlap = pd.DataFrame()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "HCD"
    df_overlap.loc[row_idx, "study_2"] = "Kang et al."
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_kang') & (scores.celltype_combo == ctc)].frac_shared_genes_1.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "Kang et al."
    df_overlap.loc[row_idx, "study_2"] = "HCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_kang') & (scores.celltype_combo == ctc)].frac_shared_genes_2.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "HCD"
    df_overlap.loc[row_idx, "study_2"] = "MCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_1.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "MCD"
    df_overlap.loc[row_idx, "study_2"] = "HCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_2.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "Kang et al."
    df_overlap.loc[row_idx, "study_2"] = "MCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'kang_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_1.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "MCD"
    df_overlap.loc[row_idx, "study_2"] = "Kang et al."
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'kang_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_2.mean()
    df_overlap.sort_values(["study_1", "study_2"])

    df_overlap.loc[:, "celltype"] = ctc
    overlaps.append(df_overlap)

df_overlap = pd.concat(overlaps, axis=0)

df_pivot = df_overlap.pivot(
    index=["study_1", "study_2"],
    columns="celltype",
    values="overlap"
)

order_y = [
    ('HCD', 'Kang et al.'),
    ('Kang et al.', 'HCD'),
    ('HCD', 'MCD'),
    ('MCD', 'HCD'),
    ('Kang et al.', 'MCD'),
    ('MCD', 'Kang et al.')
]

df_pivot = df_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(1, 1.2*3))
sns.heatmap(
    df_pivot, 
    annot=False, 
    fmt=".2f", 
    square=True, 
    cmap="Greens", 
    vmin=0, 
    vmax=1,
    cbar=False,
    linewidth=0.5,
    linecolor="white",
)

def clean_celltype_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_celltype_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        # 'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono / Monocyte",
        # 'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono / Monocyte",
        'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono",
        'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono",
        'CD4_T_cell__CD4_T_cell__T_cell_CD4': "CD4 T cell",
        'CD8_T_cell__CD8_T_cell__T_cell_CD8': "CD8 T cell", 
        # 'NK_CD56hi__NK__NK_cell': "NK CD56hi / NK cell",
        # 'NK_CD56low__NK__NK_cell': "NK CD56low / NK cell", 
        'NK_CD56hi__NK__NK_cell': "NK CD56hi",
        'NK_CD56low__NK__NK_cell': "NK CD56low", 
        'cDC__cDC__cDC1': "cDC / cDC1", 
        'cDC__cDC__cDC2': "cDC / cDC1", 
    }
    return name_map[x]

def clean_comparison_label2(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_comparison_label2(_x) for _x in x]
        
    return " vs. ".join(x)
    
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label2(df_pivot.index), fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.tick_params(length=0)
plt.xlabel("")
plt.ylabel("")
plt.title("Overlap", fontsize=6, family="Arial")
plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_overlap.svg"), pad_inches=0, bbox_inches="tight", transparent=False)

plt.xticks([])
plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_overlap_no_xlabels.svg"), pad_inches=0, bbox_inches="tight", transparent=False)

plt.show()

In [ ]:
df_pivot = scores.pivot(
    index="comparison",
    columns="celltype_combo",
    values="frac_shared_genes_1"
)
order_y = ["hcd_kang", "kang_mcd", "hcd_mcd"]
order_x = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.heatmap(
    df_pivot,
    square=True,
    cmap="Greens",
    # center=0,
    vmin=0,
    vmax=1,
    # annot=True,
    annot_kws = {"fontsize": 6, "family":"Arial"},
    cbar=False,
    linewidth=0.5,
    linecolor="white",
)
plt.xlabel("")
plt.ylabel("")

def clean_comparison_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_comparison_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        "hcd_kang": "HCD vs. Kang et al.",
        "hcd_mcd": "HCD vs. MCD",
        "kang_mcd": "MCD vs. Kang et al.",
    }
    return name_map[x]

def clean_celltype_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_celltype_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono / Monocyte",
        'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono / Monocyte",
        'CD4_T_cell__CD4_T_cell__T_cell_CD4': "CD4 T cell",
        'CD8_T_cell__CD8_T_cell__T_cell_CD8': "CD8 T cell", 
        'NK_CD56hi__NK__NK_cell': "NK CD56hi / NK cell",
        'NK_CD56low__NK__NK_cell': "NK CD56low / NK cell", 
        'cDC__cDC__cDC1': "cDC / cDC1", 
        'cDC__cDC__cDC2': "cDC / cDC1", 
    }
    return name_map[x]
    
# plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(ax.get_yticklabels()), fontsize=6, family="Arial")
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(df_pivot.index), fontsize=6, family="Arial")
# plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(ax.get_xticklabels()), fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.tick_params(length=0)
# plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_spearman.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
df_pivot = scores.pivot(
    index="comparison",
    columns="celltype_combo",
    values="frac_shared_genes_1"
)
order_y = ["hcd_kang", "kang_mcd", "hcd_mcd"]
order_x = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.heatmap(
    df_pivot,
    square=True,
    cmap="Greens",
    # center=0,
    vmin=0,
    vmax=1,
    # annot=True,
    annot_kws = {"fontsize": 6, "family":"Arial"},
    cbar=False,
    linewidth=0.5,
    linecolor="white",
)
plt.xlabel("")
plt.ylabel("")

def clean_comparison_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_comparison_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        "hcd_kang": "HCD vs. Kang et al.",
        "hcd_mcd": "HCD vs. MCD",
        "kang_mcd": "MCD vs. Kang et al.",
    }
    return name_map[x]

def clean_celltype_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_celltype_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono / Monocyte",
        'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono / Monocyte",
        'CD4_T_cell__CD4_T_cell__T_cell_CD4': "CD4 T cell",
        'CD8_T_cell__CD8_T_cell__T_cell_CD8': "CD8 T cell", 
        'NK_CD56hi__NK__NK_cell': "NK CD56hi / NK cell",
        'NK_CD56low__NK__NK_cell': "NK CD56low / NK cell", 
        'cDC__cDC__cDC1': "cDC / cDC1", 
        'cDC__cDC__cDC2': "cDC / cDC1", 
    }
    return name_map[x]
    
# plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(ax.get_yticklabels()), fontsize=6, family="Arial")
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(df_pivot.index), fontsize=6, family="Arial")
# plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(ax.get_xticklabels()), fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.tick_params(length=0)
# plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_spearman.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
df_pivot = scores.pivot(
    index="comparison",
    columns="celltype_combo",
    values="frac_shared_genes_2"
)
order_y = ["hcd_kang", "kang_mcd", "hcd_mcd"]
order_x = df_pivot.mean(axis=0).sort_values(ascending=False).index
df_pivot = df_pivot.loc[order_y, order_x]

fig, ax = plt.subplots(1,1,figsize=(1.2, 1.2))
sns.heatmap(
    df_pivot,
    square=True,
    cmap="Greens",
    # center=0,
    vmin=0,
    vmax=1,
    # annot=True,
    annot_kws = {"fontsize": 6, "family":"Arial"},
    cbar=False,
    linewidth=0.5,
    linecolor="white",
)
plt.xlabel("")
plt.ylabel("")

def clean_comparison_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_comparison_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        "hcd_kang": "HCD vs. Kang et al.",
        "hcd_mcd": "HCD vs. MCD",
        "kang_mcd": "MCD vs. Kang et al.",
    }
    return name_map[x]

def clean_celltype_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_celltype_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono / Monocyte",
        'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono / Monocyte",
        'CD4_T_cell__CD4_T_cell__T_cell_CD4': "CD4 T cell",
        'CD8_T_cell__CD8_T_cell__T_cell_CD8': "CD8 T cell", 
        'NK_CD56hi__NK__NK_cell': "NK CD56hi / NK cell",
        'NK_CD56low__NK__NK_cell': "NK CD56low / NK cell", 
        'cDC__cDC__cDC1': "cDC / cDC1", 
        'cDC__cDC__cDC2': "cDC / cDC1", 
    }
    return name_map[x]
    
# plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(ax.get_yticklabels()), fontsize=6, family="Arial")
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label(df_pivot.index), fontsize=6, family="Arial")
# plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(ax.get_xticklabels()), fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.tick_params(length=0)
# plt.savefig(os.path.join(FIGDIR, f"heatmap_comparison_kang_et_al_spearman.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="pearson",
    hue="comparison"
)
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="num_shared_genes",
    hue="comparison"
)
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="frac_shared_genes_1",
    hue="comparison"
)
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="frac_shared_genes_2",
    hue="comparison"
)
plt.show()

In [ ]:
print(f"Overlap Kang, human): {scores.loc[(scores.comparison == 'hcd_kang')].frac_shared_genes_2.mean()}")
print(f"Overlap Kang, mouse): {scores.loc[(scores.comparison == 'kang_mcd')].frac_shared_genes_1.mean()}")

In [ ]:
print(f"Overlap human, kang): {scores.loc[(scores.comparison == 'hcd_kang')].frac_shared_genes_1.mean()}")
print(f"Overlap human, mouse): {scores.loc[(scores.comparison == 'hcd_mcd')].frac_shared_genes_1.mean()}")

In [ ]:
print(f"Overlap mouse, human): {scores.loc[(scores.comparison == 'hcd_mcd')].frac_shared_genes_2.mean()}")
print(f"Overlap mouse, kang): {scores.loc[(scores.comparison == 'kang_mcd')].frac_shared_genes_2.mean()}")

In [ ]:
scores

In [ ]:
ctc = "CD14_Mono__CD14_Mono__Monocyte"
overlaps = []

for ctc in scores.celltype_combo.unique():
    df_overlap = pd.DataFrame()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "HCD"
    df_overlap.loc[row_idx, "study_2"] = "Kang et al."
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_kang') & (scores.celltype_combo == ctc)].frac_shared_genes_1.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "Kang et al."
    df_overlap.loc[row_idx, "study_2"] = "HCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_kang') & (scores.celltype_combo == ctc)].frac_shared_genes_2.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "HCD"
    df_overlap.loc[row_idx, "study_2"] = "MCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_1.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "MCD"
    df_overlap.loc[row_idx, "study_2"] = "HCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'hcd_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_2.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "Kang et al."
    df_overlap.loc[row_idx, "study_2"] = "MCD"
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'kang_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_1.mean()
    row_idx = df_overlap.shape[0]
    df_overlap.loc[row_idx, "study_1"] = "MCD"
    df_overlap.loc[row_idx, "study_2"] = "Kang et al."
    df_overlap.loc[row_idx, "overlap"] = scores.loc[(scores.comparison == 'kang_mcd') & (scores.celltype_combo == ctc)].frac_shared_genes_2.mean()
    df_overlap.sort_values(["study_1", "study_2"])

    df_overlap.loc[:, "celltype"] = ctc
    overlaps.append(df_overlap)

df_overlap = pd.concat(overlaps, axis=0)

df_pivot = df_overlap.pivot(
    index=["study_1", "study_2"],
    columns="celltype",
    values="overlap"
)

order_y = [
    ('HCD', 'Kang et al.'),
    ('Kang et al.', 'HCD'),
    ('HCD', 'MCD'),
    ('MCD', 'HCD'),
    ('Kang et al.', 'MCD'),
    ('MCD', 'Kang et al.')
]

df_pivot = df_pivot.loc[order_y, order_x]

sns.heatmap(
    df_pivot, 
    annot=True, 
    fmt=".2f", 
    square=True, 
    cmap="Greens", 
    vmin=0, 
    vmax=1,
    cbar=False
)

def clean_celltype_label(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_celltype_label(_x) for _x in x]
    if not isinstance(x, str):
        x = x.get_text()
    name_map = {
        'CD14_Mono__CD14_Mono__Monocyte': "CD14 Mono / Monocyte",
        'CD16_Mono__CD16_Mono__Monocyte': "CD16 Mono / Monocyte",
        'CD4_T_cell__CD4_T_cell__T_cell_CD4': "CD4 T cell",
        'CD8_T_cell__CD8_T_cell__T_cell_CD8': "CD8 T cell", 
        'NK_CD56hi__NK__NK_cell': "NK CD56hi / NK cell",
        'NK_CD56low__NK__NK_cell': "NK CD56low / NK cell", 
        'cDC__cDC__cDC1': "cDC / cDC1", 
        'cDC__cDC__cDC2': "cDC / cDC1", 
    }
    return name_map[x]

def clean_comparison_label2(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [clean_comparison_label2(_x) for _x in x]
        
    return " vs. ".join(x)
    
plt.yticks(0.5+np.arange(df_pivot.shape[0]), clean_comparison_label2(df_pivot.index), fontsize=6, family="Arial")
plt.xticks(0.5+np.arange(df_pivot.shape[1]), clean_celltype_label(df_pivot.columns), fontsize=6, family="Arial")
plt.tick_params(length=0)
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
sns.heatmap(df_overlap.set_index(["study_1", "study_2"]), square=True, cmap="Greens", vmin=0, vmax=1, cbar=False, annot=True)
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="num_genes_1",
    hue="comparison"
)
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="num_genes_2",
    hue="comparison"
)
plt.show()

In [ ]:
sns.barplot(
    data=scores,
    y="celltype_combo",
    x="jaccard",
    hue="comparison"
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(2,2))
ax.plot(lfc_hcd, lfc_kang, "o", ms=2)
xx = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1])
ax.plot(xx, xx, "--k")
plt.xlabel("human cyto dict")
plt.ylabel("Kang et al.")
plt.show()

fig, ax = plt.subplots(1,1,figsize=(2,2))
ax.plot(lfc_mcd, lfc_kang, "o", ms=2)
xx = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1])
ax.plot(xx, xx, "--k")
plt.xlabel("mouse cyto dict")
plt.ylabel("Kang et al.")
plt.show()

fig, ax = plt.subplots(1,1,figsize=(2,2))
ax.plot(lfc_hcd, lfc_mcd, "o", ms=2)
xx = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1])
ax.plot(xx, xx, "--k")
plt.xlabel("human cyto dict")
plt.ylabel("mouse cyto dict")
plt.show()